# 03 — Classical ML baselines
Compare four classical classifiers using training-only cross-validation. The holdout set is reserved for final evaluation.

In [ ]:
%pip -q install scikit-learn joblib

In [ ]:
from pathlib import Path
import os, json, time, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, StratifiedKFold, cross_validate
from sklearn.impute import SimpleImputer, KNNImputer
from sklearn.experimental import enable_iterative_imputer
from sklearn.impute import IterativeImputer
from sklearn.preprocessing import StandardScaler, MinMaxScaler
from sklearn.pipeline import Pipeline
from sklearn.feature_selection import mutual_info_classif, SelectKBest, RFE
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score,
    fbeta_score, roc_auc_score, average_precision_score, confusion_matrix,
    classification_report, roc_curve, precision_recall_curve)
warnings.filterwarnings("ignore")
RANDOM_STATE = 42
PROJECT_ROOT = Path.cwd()
# If running from a notebook in /notebooks, move to project root.
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
DATA_PATH = PROJECT_ROOT / "dataset" / "diabetes.csv"
if not DATA_PATH.exists():
    # Colab fallback: upload diabetes.csv into the current session or mount Drive.
    DATA_PATH = Path("/content/diabetes.csv")
df = pd.read_csv(DATA_PATH)
TARGET = "Outcome"
FEATURES = [c for c in df.columns if c != TARGET]
INVALID_ZERO_COLS = ["Glucose", "BloodPressure", "SkinThickness", "Insulin", "BMI"]

from sklearn.metrics import make_scorer
from sklearn.model_selection import cross_validate
X=df.drop(columns=TARGET).replace({c:{0:np.nan} for c in INVALID_ZERO_COLS})
y=df[TARGET].astype(int)
models={
 "Logistic Regression": LogisticRegression(max_iter=2000,class_weight="balanced"),
 "Random Forest": RandomForestClassifier(n_estimators=300,min_samples_leaf=2,class_weight="balanced",random_state=42),
 "RBF-SVM": SVC(kernel="rbf",C=1.0,gamma="scale",class_weight="balanced",probability=True,random_state=42),
 "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
cv=StratifiedKFold(n_splits=5,shuffle=True,random_state=42)
rows=[]
for name,model in models.items():
    steps=[("imputer",SimpleImputer(strategy="median"))]
    if name in ["Logistic Regression","RBF-SVM"]: steps.append(("scaler",StandardScaler()))
    pipe=Pipeline(steps+[("model",model)])
    scores=cross_validate(pipe,X,y,cv=cv,scoring={"accuracy":"accuracy","recall":"recall","f1":"f1","roc_auc":"roc_auc"},n_jobs=-1)
    rows.append({"model":name,**{f"{m}_mean":scores[f"test_{m}"].mean() for m in ["accuracy","recall","f1","roc_auc"]},
                 **{f"{m}_std":scores[f"test_{m}"].std() for m in ["accuracy","recall","f1","roc_auc"]}})
results=pd.DataFrame(rows)
display(results.sort_values("recall_mean",ascending=False))
results.to_csv("../results/classical_cv.csv",index=False)

## Evaluation note
These are cross-validation estimates. Do not call them test-set results. Final holdout evaluation and threshold selection will be handled separately.